# Lecture 3 - Python essentials II

**Lecture Topic**: Reading and debugging Python: comprehensions, exceptions and tracebacks, modules and scripts, and regular expressions

**Instructor**: Duncan Watson-Parris

## Announcement

The first *weekly quiz* is now available on Canvas. It's only short and is due on Sunday, October 4th at 11:59pm.

The questions are based on the material covered Monday and today.

ClassBuzz poll for today: `TJY3`

## Recap

In the last lecture, we covered the following topics:
 - DataHub and Git
 - Variables, types and strings
 - Lists, tuples, dictionaries and sets
 - Mutability
 - Control flow

Today, we will cover the following topics:
 - Functions
 - Comprehensions, lambdas and generators
 - Exceptions and tracebacks
 - Files, modules and scripts

We'll start by finishing off functions from Monday. After that, today is more about reading Python than writing it. These are all things you'll see constantly in other people's code, and in code an agent writes for you.

## Functions

Functions help improve readability and usability by encapsulating code.

- Functions hide details of what exactly is going on inside a program.
- Functions consist of parameters, a function body, and a return expression.

In [ ]:
def example_function():
    return "This is an example function"

print(example_function())

### Parameters vs Arguments

Parameters are the inputs defined by the function, while arguments are the actual values you pass in when calling the function.

In [ ]:
def discriminant(a, b, c):
    return b**2 - 4*a*c

# Function call with arguments
discriminant(1, 2, 3)

We can provide default values for parameters, which allows us to call the function without providing that parameter.

It is also possible to pass arguments by name, which allows you to pass arguments in any order:

In [ ]:
def discriminant(a=0, b=0, c=0):
    return b**2 - 4*a*c

print(discriminant(1, 3))
print(discriminant(c=1, b=2))

## Passing a Variable Number of Arguments

`*args` is used to pass a variable number of non-keyword arguments to a function, `**kwargs` is used to pass a variable number of keyword arguments to a function:

In [ ]:
def my_function(*args, **kwargs):
    print(args)
    print(kwargs)

my_function(2, 3, a=4, b=5)

This is particularly useful when you are writing a function that wraps another function and you want to pass all (or most) of the arguments to the wrapped function. You'll see it a lot in library code.

It's good practice to provide a docstring for your functions, which describes what the function does and what parameters it expects.

In [ ]:
def discriminant(a=0, b=0, c=0):
    """
    This function returns the discriminant of a quadratic equation.

    Parameters:
    a (int): Coefficient of x^2, default is 0
    b (int): Coefficient of x, default is 0
    c (int): Constant, default is 0
    """
    return b**2 - 4*a*c

help(discriminant)

Agents are generally very good at writing docstrings. Do read them, but check them against what the code actually does. They don't always agree!

### Quick quiz: spot the bug

Here's a function to collect temperature readings from a weather station. What does the last line print?

 - A. `[19.0]`
 - B. `[21.5, 19.0]`
 - C. `[]`
 - D. An error

In [ ]:
def add_reading(value, readings=[]):
    readings.append(value)
    return readings

station_a = add_reading(21.5)
station_b = add_reading(19.0)
print(station_b)

Default values are evaluated once, when the function is defined, not each time it is called. So every call that uses the default shares **the same list**.

This is mutability again, and it's a nasty one: the code runs without any errors, and a test that only calls it once will pass.

The usual fix is to use `None` as the default:

In [ ]:
def add_reading(value, readings=None):
    if readings is None:
        readings = []
    readings.append(value)
    return readings

station_a = add_reading(21.5)
station_b = add_reading(19.0)
print(station_b)

## Comprehensions

A list comprehension is a compact way of building a list from another iterable:

In [ ]:
squares = [x**2 for x in range(10)]
squares

This is equivalent to:

In [ ]:
squares = []
for x in range(10):
    squares.append(x**2)

You can add a condition, and the same syntax works for dictionaries and sets:

In [ ]:
evens = [x for x in range(10) if x % 2 == 0]
lengths = {name: len(name) for name in ["numpy", "pandas", "xarray"]}
remainders = {x % 3 for x in range(10)}

print(evens)
print(lengths)
print(remainders)

### Quick quiz

What is the output of the following code?

 - A. `['TAXI', 'FARE']`
 - B. `['TIP', 'TOTAL']`
 - C. `['TAXI', 'TIP', 'FARE', 'TOTAL']`
 - D. `['taxi', 'fare']`

In [ ]:
words = ["taxi", "Tip", "fare", "Total"]
[w.upper() for w in words if w[0].islower()]

Comprehensions are great for short transformations, but they quickly become unreadable when they get nested. If you can't read one in a few seconds, it's fine to ask for (or write) a loop instead.

## Lambda Functions

Lambda functions are small, anonymous functions that can have any number of arguments, but only one expression.

In [ ]:
add = lambda x, y: x + y
print(add(2, 3))

You'll mostly see them passed to other functions, for example as the key to sort by:

In [ ]:
trips = [("JFK", 52.0), ("Midtown", 9.5), ("Harlem", 14.25)]
sorted(trips, key=lambda trip: trip[1])

### Map and filter

`map()` applies a function to each item in an iterable, and `filter()` keeps the items for which the function returns `True`:

In [ ]:
numbers = [1, 2, 3, 4, 5]
print(list(map(lambda x: x**2, numbers)))
print(list(filter(lambda x: x % 2 == 0, numbers)))

Note, we use `list()` to convert the result to a list, as `map()` and `filter()` return iterators.

These are not necessarily quicker than a list comprehension, and the comprehension is usually easier to read.

## Generators

A generator produces its values one at a time, as they are asked for, rather than all at once:

In [ ]:
big = (x * 10 for x in range(3_000_000))   # note the round brackets
print(next(big))
print(next(big))

Generators are useful when you need to generate a large number of values, but you don't want to store them all in memory at once.

### Quick quiz

What does this print?

In [ ]:
squares = (x**2 for x in range(5))
print(sum(squares))
print(sum(squares))

A generator can only be used once. The second `sum` gets an empty generator, and adds up nothing without complaint.

## Exception handling

You will have noticed that when something goes wrong in a Python program you see an error message.

This is called an exception, and you can handle them explicitly to prevent your program from aborting and printing an unhelpful traceback.

In [ ]:
value = "5 trips"
n = int(value)

We can catch the exception and do something more helpful instead:

In [ ]:
try:
    n = int(value)
except ValueError:
    print(f"Couldn't convert {value!r} to an integer")

## Reading a traceback

Before you handle an exception, you need to understand it. Here is some code that reads the fare from a list of taxi trips:

In [ ]:
def parse_fare(row):
    return float(row["fare"])

def load_fares(rows):
    return [parse_fare(r) for r in rows]

rows = [{"fare": "12.5"}, {"fare": "8.0"}, {"Fare": "30.0"}]
load_fares(rows)

Tracebacks are read from the **bottom up**:

- The last line tells you what went wrong: a `KeyError`, for the key `'fare'`.
- The lines above tell you how you got there, with the most recent call last.

Notice what it doesn't tell you though: which row was the problem. For that you need to go and look at the data.

### Quick quiz: is this a good fix?

In [ ]:
def parse_fare(row):
    try:
        return float(row["fare"])
    except Exception:
        return 0.0

load_fares(rows)

This runs without errors. The missing fare has become a fare of zero, and any average you compute from it is now wrong.

It's a common pattern in agent-written code, because it makes the error go away. When you see a broad `except`, ask what happens to the data that ends up in it.

Better options are to catch only the specific exception you expect, and either return something that can't be mistaken for real data (like `None`), or let it fail loudly.

## Files and context managers

Python has a built-in function `open()` to open a file. It is almost always used with the `with` statement:

In [ ]:
with open("test.txt", "w") as f:
    f.write("Hello World\n")

with open("test.txt") as f:
    print(f.read())

This takes care of closing the file for you, even if an exception is raised.

For csv and other structured data, you can use `pandas` to read the file into a DataFrame and it will automatically parse the data into columns (and is much faster than doing it manually).

## Libraries and Modules

Python has a very rich standard library, which includes modules for working with files, regular expressions, dates and times, compression, and much more.
You can also install third-party libraries from PyPI using `pip`, or from conda-forge using `conda`.

To use a module, you need to import it:

```python
import os                  # the whole module
from os import path        # one thing from a module
import numpy as np         # a module with an alias
```

Note, when an agent imports a package, check that it actually exists and that you've heard of it. Agents occasionally invent plausible-sounding package names, and someone may have registered that name with something malicious. We'll come back to this in the security lecture.

### Scripts vs notebooks

Notebooks are fantastic for learning and teaching, as they allow you to write code and see the output immediately.
For more intensive analysis code, you may want to use a Python script, which can be run from the command line.

Any `.py` file is a module you can import, and also a script you can run:

In [ ]:
%%writefile fares.py
def parse_fare(row):
    """Return the fare of a trip as a float."""
    return float(row["fare"])

if __name__ == "__main__":
    print(parse_fare({"fare": "12.5"}))

In [ ]:
!python fares.py

In [ ]:
import fares
fares.parse_fare({"fare": "8.0"})

The code under `if __name__ == "__main__":` only runs when the file is executed directly, not when it is imported.

### A word of warning about notebooks

Cells can be run in any order, and the notebook remembers everything you've run, even cells you've since deleted.

In [ ]:
threshold = 10

In [ ]:
trip_fares = [5, 12, 30]
[f for f in trip_fares if f > threshold]

Before trusting a notebook (yours or anyone else's), do **Kernel ▸ Restart & Run All** and check that you get the same answer.

## Summary

Today, we covered the following topics:
 - Functions (and why you shouldn't use a list as a default)
 - Comprehensions, lambdas and generators
 - Exceptions, and how to read a traceback
 - Files, modules and scripts

On Friday, regular expressions and NumPy.